In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Read from Silver (batch semantics)
# -------------------------
def _silver_obs():
    # Use dlt.read (not read_stream) so Gold is computed as a batch snapshot per run (Triggered mode)
    return dlt.read("canada_business.silver.silver_fred_observations")

def _silver_runs():
    return dlt.read("canada_business.silver.silver_fred_runs")


# ------------------------------------------------
# GOLD 0: Daily fact (BI-friendly copy of Silver)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_daily",
    comment="Daily FRED observations (clean, BI-ready). One row per series_id per day."
)
def gold_fred_daily():
    df = _silver_obs()

    return (
        df.select(
            "series_id",
            F.col("observation_date").cast("date").alias("observation_date"),
            F.col("value_double").cast("double").alias("value"),
            F.col("value_raw").cast("string").alias("value_raw"),
            F.col("_ingest_ts").alias("ingest_ts"),
            F.col("_source_file").alias("source_file"),
            F.col("realtime_start").cast("date").alias("realtime_start"),
            F.col("realtime_end").cast("date").alias("realtime_end"),
            F.col("obs_realtime_start").cast("date").alias("obs_realtime_start"),
            F.col("obs_realtime_end").cast("date").alias("obs_realtime_end"),
        )
        .where(F.col("series_id").isNotNull() & F.col("observation_date").isNotNull())
        .dropDuplicates(["series_id", "observation_date"])  # <-- ADD THIS
    )


# ------------------------------------------------
# GOLD 1: Latest value per series (dashboard KPI)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_latest",
    comment="Latest available value per series (for KPI tiles / overview dashboards)."
)
def gold_fred_latest():
    df = dlt.read("gold_fred_daily")

    # Get latest date per series
    latest_dates = (
        df.groupBy("series_id")
          .agg(F.max("observation_date").alias("latest_date"))
    )

    # Join back to get the corresponding row
    latest = (
        df.join(latest_dates, on="series_id", how="inner")
          .where(F.col("observation_date") == F.col("latest_date"))
          .drop("latest_date")
    )

    return latest


# ------------------------------------------------
# GOLD 2: Monthly aggregates (BI time grain)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_monthly",
    comment="Monthly aggregates per series (avg/min/max/last)."
)
def gold_fred_monthly():
    df = dlt.read("gold_fred_daily")

    m = df.withColumn("year_month", F.date_trunc("month", F.col("observation_date")).cast("date"))

    # "last_value_in_month" using max_by on observation_date (Spark supports max_by)
    return (
        m.groupBy("series_id", "year_month")
         .agg(
             F.avg("value").alias("avg_value"),
             F.min("value").alias("min_value"),
             F.max("value").alias("max_value"),
             F.max_by(F.col("value"), F.col("observation_date")).alias("last_value_in_month"),
             F.count("*").alias("days_in_month")
         )
    )


# ------------------------------------------------
# GOLD 3: YoY % change (on monthly avg)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_yoy",
    comment="Year-over-year % change computed from monthly avg_value."
)
def gold_fred_yoy():
    df = dlt.read("gold_fred_monthly")

    w = Window.partitionBy("series_id").orderBy(F.col("year_month"))

    return (
        df.withColumn("avg_value_prev_year", F.lag("avg_value", 12).over(w))
          .withColumn(
              "yoy_pct",
              F.when(F.col("avg_value_prev_year").isNull(), F.lit(None).cast("double"))
               .when(F.col("avg_value_prev_year") == 0, F.lit(None).cast("double"))
               .otherwise((F.col("avg_value") - F.col("avg_value_prev_year")) / F.col("avg_value_prev_year") * 100.0)
          )
          .select("series_id", "year_month", "avg_value", "avg_value_prev_year", "yoy_pct")
    )


# ------------------------------------------------
# GOLD 4: Rolling 12-month average (monthly)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_rolling_12m",
    comment="Rolling 12-month average of monthly avg_value (smooth trend)."
)
def gold_fred_rolling_12m():
    df = dlt.read("gold_fred_monthly")

    w = (
        Window.partitionBy("series_id")
              .orderBy(F.col("year_month"))
              .rowsBetween(-11, 0)
    )

    return (
        df.withColumn("rolling_12m_avg", F.avg("avg_value").over(w))
          .select("series_id", "year_month", "avg_value", "rolling_12m_avg")
    )


# ------------------------------------------------
# GOLD 5: 30-day volatility (daily stddev)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_volatility_30d",
    comment="30-day rolling volatility (stddev) on daily values per series."
)
def gold_fred_volatility_30d():
    df = dlt.read("gold_fred_daily")

    w = (
        Window.partitionBy("series_id")
              .orderBy(F.col("observation_date"))
              .rowsBetween(-29, 0)
    )

    return (
        df.select("series_id", "observation_date", "value")
          .withColumn("volatility_30d", F.stddev_samp("value").over(w))
    )


# ------------------------------------------------
# GOLD 6: Run / audit table (carry forward)
# ------------------------------------------------
@dlt.table(
    name="gold_fred_runs",
    comment="Gold copy of run audit derived from manifest (good for lineage & dashboards)."
)
def gold_fred_runs():
    return _silver_runs()

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def week_ending_sunday(dcol):
    return F.next_day(F.date_sub(dcol, 1), "Sun")

@dlt.table(
  name="gold_macro_demand_weekly",
  comment="Weekly macro + demand with forward-filled macro values"
)
def gold_macro_demand_weekly():

    fred = dlt.read("canada_business.silver.silver_fred_observations")

    series_map = {
        "DCOILWTICO": "oil_wti",
        "DGS10": "rate_10y",
        "FEDFUNDS": "rate_fedfunds",
        "T5YIE": "infl_breakeven_5y",
        "ICSA": "claims_initial",
        "IC4WSA": "claims_4wma",
        "DEXCAUS": "fx_cad_usd",
        "FRGSHPUSM649NCIS": "freight_cass",
        "WPU1171": "ppi_electronics",
        "MRTSSM443USS": "retail_elec_sales",
        "UMCSENT": "consumer_sentiment",
        "HOUST": "housing_starts",
        "DSPIC96": "real_disposable_income"
    }

    fred_w = (
        fred.where(F.col("series_id").isin(list(series_map.keys())))
            .withColumn("week_end_sun", week_ending_sunday(F.col("observation_date")))
            .groupBy("week_end_sun")
            .agg(
                *[
                    F.avg(F.when(F.col("series_id") == sid, F.col("value_double"))).alias(alias)
                    for sid, alias in series_map.items()
                ]
            )
            .withColumnRenamed("week_end_sun", "date")
    )

    trends = dlt.read("canada_business.silver.silver_google_trends")

    joined = fred_w.join(trends, on="date", how="left")

    # ---- Forward fill all macro columns ----
    window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

    macro_cols = list(series_map.values())

    for c in macro_cols:
        joined = joined.withColumn(
            c,
            F.last(F.col(c), ignorenulls=True).over(window_spec)
        )

    return joined.withColumn("ingest_ts", F.current_timestamp())

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================
# WITS Tariff Gold Layer
# ============================================================================
# Industry techniques demonstrated:
#   1. Analytical pivoting (long → wide for BI consumption)
#   2. Window functions for YoY change calculation
#   3. Latest-value snapshot for KPI dashboards
#   4. Product-category filtering for domain-specific views
#   5. Forward-fill not needed here (annual data, no gaps within series)
# ============================================================================


# ------------------------------------------------
# GOLD 1: Clean tariff fact table (BI-ready)
# ------------------------------------------------
@dlt.table(
    name="gold_wits_tariff_annual",
    comment="Annual WITS tariff rates by product category — BI-ready fact table"
)
def gold_wits_tariff_annual():
    """
    TECHNIQUE: BI-ready fact table
    One row per reporter/partner/product/indicator/year with clean column names.
    Serves as the single source of truth for all tariff analytics downstream.
    """
    df = dlt.read("canada_business.silver.silver_wits_tariff")

    return (
        df.select(
            "reporter",
            "partner",
            "productcode",
            "indicator",
            F.col("year").cast("int").alias("year"),
            F.col("tariff_value").cast("double").alias("tariff_rate_pct"),
            "freq",
            "series_key",
            "source_file",
            "ingest_ts"
        )
        .where(F.col("year").isNotNull() & F.col("tariff_rate_pct").isNotNull())
    )


# ------------------------------------------------
# GOLD 2: Latest tariff per product (KPI tiles)
# ------------------------------------------------
@dlt.table(
    name="gold_wits_tariff_latest",
    comment="Latest available tariff rate per product category (for KPI tiles / overview dashboards)"
)
def gold_wits_tariff_latest():
    """
    TECHNIQUE: Latest-value snapshot
    For each product category, return only the most recent year's tariff rate.
    Powers executive KPI dashboards without requiring date filters.
    """
    df = dlt.read("gold_wits_tariff_annual")

    latest_years = (
        df.groupBy("reporter", "partner", "productcode", "indicator")
          .agg(F.max("year").alias("latest_year"))
    )

    return (
        df.join(latest_years,
                on=["reporter", "partner", "productcode", "indicator"],
                how="inner")
          .where(F.col("year") == F.col("latest_year"))
          .drop("latest_year")
    )


# ------------------------------------------------
# GOLD 3: YoY change in tariff rates
# ------------------------------------------------
@dlt.table(
    name="gold_wits_tariff_yoy",
    comment="Year-over-year change in weighted average tariff rates per product category"
)
def gold_wits_tariff_yoy():
    """
    TECHNIQUE: Window-based YoY calculation
    Uses LAG(1) over year (not LAG(12) like monthly data) since tariff data
    is annual. Computes both absolute change and percentage change for
    flexible dashboard use.
    """
    df = dlt.read("gold_wits_tariff_annual")

    w = Window.partitionBy("reporter", "partner", "productcode", "indicator").orderBy("year")

    return (
        df.withColumn("prev_year_rate", F.lag("tariff_rate_pct", 1).over(w))
          .withColumn("prev_year", F.lag("year", 1).over(w))
          .withColumn(
              "yoy_abs_change",
              F.col("tariff_rate_pct") - F.col("prev_year_rate")
          )
          .withColumn(
              "yoy_pct_change",
              F.when(F.col("prev_year_rate").isNull(), F.lit(None).cast("double"))
               .when(F.col("prev_year_rate") == 0, F.lit(None).cast("double"))
               .otherwise(
                   (F.col("tariff_rate_pct") - F.col("prev_year_rate"))
                   / F.col("prev_year_rate") * 100.0
               )
          )
          .select(
              "reporter", "partner", "productcode", "indicator",
              "year", "tariff_rate_pct", "prev_year", "prev_year_rate",
              "yoy_abs_change", "yoy_pct_change"
          )
    )


# ------------------------------------------------
# GOLD 4: Pivoted view — tariff by major sector
# ------------------------------------------------
@dlt.table(
    name="gold_wits_tariff_by_sector",
    comment="Pivoted tariff rates for key sectors (wide format for dashboards)"
)
def gold_wits_tariff_by_sector():
    """
    TECHNIQUE: Analytical pivoting (long → wide)
    Transforms the normalized long-format data into a wide BI-friendly layout
    with one column per product sector. This eliminates the need for pivot
    operations in downstream BI tools like Power BI or Tableau.
    """
    df = dlt.read("gold_wits_tariff_annual")

    # Map of productcode → friendly column alias
    sectors = {
        "Total":            "all_products",
        "01-05_Animal":     "animal",
        "16-24_FoodProd":   "food_products",
        "27-27_Fuels":      "fuels",
        "28-38_Chemicals":  "chemicals",
        "39-40_PlastiRub":  "plastic_rubber",
        "50-63_TextCloth":  "textiles_clothing",
        "72-83_Metals":     "metals",
        "84-85_MachElec":   "machinery_electronics",
        "86-89_Transport":  "transportation",
        "manuf":            "manufactures",
    }

    pivot_aggs = [
        F.max(F.when(F.col("productcode") == code, F.col("tariff_rate_pct")))
         .alias(alias)
        for code, alias in sectors.items()
    ]

    return (
        df.where(F.col("productcode").isin(list(sectors.keys())))
          .groupBy("reporter", "partner", "indicator", "year")
          .agg(*pivot_aggs)
    )


# ------------------------------------------------
# GOLD 5: Electronics-specific tariff trend
# ------------------------------------------------
@dlt.table(
    name="gold_wits_tariff_electronics",
    comment="Tariff trend for Machinery & Electronics (84-85) — key category for Canada electronics business"
)
def gold_wits_tariff_electronics():
    """
    TECHNIQUE: Domain-specific filtered view
    Pre-filters to the product category most relevant to this project's
    business question (Canada electronics retail). Avoids forcing analysts
    to filter repeatedly in dashboards.
    """
    df = dlt.read("gold_wits_tariff_annual")

    w = Window.partitionBy("reporter", "partner", "indicator").orderBy("year")

    return (
        df.where(F.col("productcode") == "84-85_MachElec")
          .withColumn("prev_year_rate", F.lag("tariff_rate_pct", 1).over(w))
          .withColumn(
              "yoy_pct_change",
              F.when(F.col("prev_year_rate").isNull(), F.lit(None).cast("double"))
               .when(F.col("prev_year_rate") == 0, F.lit(None).cast("double"))
               .otherwise(
                   (F.col("tariff_rate_pct") - F.col("prev_year_rate"))
                   / F.col("prev_year_rate") * 100.0
               )
          )
          # ---- TECHNIQUE: Rolling average for trend smoothing ----
          .withColumn(
              "rolling_3yr_avg",
              F.avg("tariff_rate_pct").over(
                  Window.partitionBy("reporter", "partner", "indicator")
                        .orderBy("year")
                        .rowsBetween(-2, 0)
              )
          )
          .select(
              "reporter", "partner", "productcode", "indicator",
              "year", "tariff_rate_pct", "prev_year_rate",
              "yoy_pct_change", "rolling_3yr_avg"
          )
    )

In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================
# StatCan Gold Layer
# ============================================================================
# Industry techniques demonstrated:
#   1. Monthly aggregation with MoM and YoY calculations
#   2. Inventory health classification (business rule engine)
#   3. Import pipeline trend with rolling averages
#   4. Combined supply-side dashboard view
#   5. Forward-fill for sparse monthly government data
# ============================================================================


# ------------------------------------------------
# GOLD 1: Electronics Retail Monthly Trend
# ------------------------------------------------
@dlt.table(
    name="gold_statcan_retail_monthly",
    comment="Monthly electronics retail sales with MoM and YoY change"
)
def gold_statcan_retail_monthly():
    """
    TECHNIQUE: Monthly trend with lag-based change calculations
    Computes month-over-month and year-over-year changes from
    StatCan retail data. Uses seasonally adjusted values when
    available to avoid holiday-driven noise.
    """
    df = dlt.read("canada_business.silver.silver_statcan_retail_electronics")

    w_mom = Window.partitionBy("naics_label", "adjustment_type").orderBy("observation_date")

    return (
        df.groupBy("observation_date", "naics_label", "adjustment_type")
          .agg(
              F.sum("retail_sales_cad").alias("retail_sales_cad"),
              F.first("unit_of_measure").alias("unit_of_measure"),
              F.first("scalar_factor").alias("scalar_factor"),
          )
          .withColumn("prev_month_sales", F.lag("retail_sales_cad", 1).over(w_mom))
          .withColumn("prev_year_sales",  F.lag("retail_sales_cad", 12).over(w_mom))
          .withColumn(
              "mom_pct_change",
              F.when(F.col("prev_month_sales").isNull() | (F.col("prev_month_sales") == 0),
                     F.lit(None).cast("double"))
               .otherwise(
                   (F.col("retail_sales_cad") - F.col("prev_month_sales"))
                   / F.col("prev_month_sales") * 100.0
               )
          )
          .withColumn(
              "yoy_pct_change",
              F.when(F.col("prev_year_sales").isNull() | (F.col("prev_year_sales") == 0),
                     F.lit(None).cast("double"))
               .otherwise(
                   (F.col("retail_sales_cad") - F.col("prev_year_sales"))
                   / F.col("prev_year_sales") * 100.0
               )
          )
          .withColumn(
              "rolling_3m_avg",
              F.avg("retail_sales_cad").over(
                  w_mom.rowsBetween(-2, 0)
              )
          )
    )


# ------------------------------------------------
# GOLD 2: Inventory Health Classification
# ------------------------------------------------
@dlt.table(
    name="gold_statcan_inventory_health",
    comment="Monthly inventory-to-sales ratio with health classification"
)
def gold_statcan_inventory_health():
    inv = dlt.read("canada_business.silver.silver_statcan_inventory_ratio")
    sales = dlt.read("canada_business.silver.silver_statcan_retail_electronics")

    inv_agg = (
        inv.where(
            F.col("naics_label").contains("Computer and communications") |
            F.col("naics_label").contains("Home entertainment")
        )
        .where(F.col("adjustment_type") == "Seasonally adjusted")
        .groupBy("observation_date")
        .agg(F.sum("inventory_sales_ratio").alias("inventory_cad"))
    )

    sales_agg = (
        sales.where(F.col("adjustment_type").contains("Seasonally adjusted"))
        .groupBy("observation_date")
        .agg(F.sum("retail_sales_cad").alias("sales_cad"))
    )

    w = Window.orderBy("observation_date")

    joined = (
        inv_agg.join(sales_agg, on="observation_date", how="inner")
        .withColumn(
            "inventory_sales_ratio",
            F.when(F.col("sales_cad") == 0, F.lit(None))
             .otherwise(F.col("inventory_cad") / F.col("sales_cad"))
        )
        .withColumn("adjustment_type", F.lit("Seasonally adjusted"))
        .withColumn("naics_label", F.lit("Electronics (Computer & Home Entertainment)"))
        .withColumn("unit_of_measure", F.lit("Ratio"))
        .withColumn(
            "health_status",
            F.when(F.col("inventory_sales_ratio") < 0.8, "critical_low")
             .when(F.col("inventory_sales_ratio") < 1.0, "low")
             .when(F.col("inventory_sales_ratio") <= 1.5, "healthy")
             .when(F.col("inventory_sales_ratio") <= 2.0, "high")
             .otherwise("critical_high")
        )
        .withColumn(
            "recommendation",
            F.when(F.col("health_status") == "critical_low", "emergency_reorder")
             .when(F.col("health_status") == "low", "increase_purchasing")
             .when(F.col("health_status") == "healthy", "maintain_current")
             .when(F.col("health_status") == "high", "reduce_purchasing")
             .otherwise("liquidate_excess")
        )
        .withColumn("prev_month_ratio", F.lag("inventory_sales_ratio", 1).over(w))
        .withColumn("prev_year_ratio", F.lag("inventory_sales_ratio", 12).over(w))
        .withColumn("mom_change", F.col("inventory_sales_ratio") - F.col("prev_month_ratio"))
        .withColumn("yoy_change", F.col("inventory_sales_ratio") - F.col("prev_year_ratio"))
        .withColumn("rolling_6m_avg", F.avg("inventory_sales_ratio").over(w.rowsBetween(-5, 0)))
    )

    return joined


# ------------------------------------------------
# GOLD 3: Electronics Import Pipeline Monthly
# ------------------------------------------------
@dlt.table(
    name="gold_statcan_imports_monthly",
    comment="Monthly aggregated electronics imports with trend indicators"
)
def gold_statcan_imports_monthly():
    """
    TECHNIQUE: Import pipeline aggregation
    Aggregates across all NAPCS electronics subcategories to get
    a single monthly import volume. This is a leading indicator:
    if imports drop, inventory shortages may follow in 1-3 months.
    """
    df = dlt.read("canada_business.silver.silver_statcan_imports_electronics")

    w = Window.orderBy("observation_date")

    agg = (
        df.groupBy("observation_date")
          .agg(
              F.sum("import_value_cad").alias("total_import_value_cad"),
              F.countDistinct("napcs_label").alias("napcs_categories_count"),
              F.first("unit_of_measure").alias("unit_of_measure"),
          )
    )

    return (
        agg.withColumn("prev_month_imports", F.lag("total_import_value_cad", 1).over(w))
           .withColumn("prev_year_imports",  F.lag("total_import_value_cad", 12).over(w))
           .withColumn(
               "mom_pct_change",
               F.when(F.col("prev_month_imports").isNull() | (F.col("prev_month_imports") == 0),
                      F.lit(None).cast("double"))
                .otherwise(
                    (F.col("total_import_value_cad") - F.col("prev_month_imports"))
                    / F.col("prev_month_imports") * 100.0
                )
           )
           .withColumn(
               "yoy_pct_change",
               F.when(F.col("prev_year_imports").isNull() | (F.col("prev_year_imports") == 0),
                      F.lit(None).cast("double"))
                .otherwise(
                    (F.col("total_import_value_cad") - F.col("prev_year_imports"))
                    / F.col("prev_year_imports") * 100.0
                )
           )
           .withColumn(
               "rolling_3m_avg",
               F.avg("total_import_value_cad").over(w.rowsBetween(-2, 0))
           )
    )


In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================
# CAPSTONE: Unified ML Training Dataset
# ============================================================================
# Joins all 4 data domains (FRED macro, Google Trends demand, StatCan supply,
# WITS tariffs) into a single monthly feature table for ML model training.
# ============================================================================

@dlt.table(
    name="gold_ml_training_dataset",
    comment="Unified monthly feature table joining macro, demand, supply, and tariff signals"
)
def gold_ml_training_dataset():

    # ---- 1. FRED macro (monthly avg per series, pivoted wide) ----
    fred = dlt.read("gold_fred_monthly")

    fred_series = {
        "DCOILWTICO": "oil_wti",
        "DGS10": "rate_10y",
        "FEDFUNDS": "rate_fedfunds",
        "T5YIE": "infl_breakeven_5y",
        "ICSA": "claims_initial",
        "IC4WSA": "claims_4wma",
        "DEXCAUS": "fx_cad_usd",
        "FRGSHPUSM649NCIS": "freight_cass",
        "WPU1171": "ppi_electronics",
        "MRTSSM443USS": "retail_elec_sales",
        "UMCSENT": "consumer_sentiment",
        "HOUST": "housing_starts",
        "DSPIC96": "real_disposable_income",
    }

    fred_pivot = (
        fred.where(F.col("series_id").isin(list(fred_series.keys())))
            .groupBy("year_month")
            .agg(
                *[
                    F.max(F.when(F.col("series_id") == sid, F.col("avg_value"))).alias(alias)
                    for sid, alias in fred_series.items()
                ]
            )
            .withColumnRenamed("year_month", "month_date")
    )

    # ---- 2. Google Trends demand (aggregate weekly -> monthly) ----
    trends = dlt.read("canada_business.silver.silver_google_trends")

    trends_monthly = (
        trends.withColumn("month_date", F.date_trunc("month", F.col("date")).cast("date"))
              .groupBy("month_date")
              .agg(
                  F.avg("laptop_idx").alias("laptop_idx"),
                  F.avg("tv_idx").alias("tv_idx"),
                  F.avg("console_idx").alias("console_idx"),
                  F.avg("electronics_sale_idx").alias("electronics_sale_idx"),
                  F.avg("demand_trend_score").alias("demand_trend_score"),
              )
    )

    # ---- 3. StatCan supply (retail + inventory + imports) ----
    retail = dlt.read("gold_statcan_retail_monthly")
    retail_agg = (
        retail.where(F.col("adjustment_type").contains("Seasonally adjusted"))
              .withColumn("month_date", F.col("observation_date").cast("date"))
              .groupBy("month_date")
              .agg(F.sum("retail_sales_cad").alias("ca_retail_elec_sales"))
    )

    inventory = dlt.read("gold_statcan_inventory_health")
    inventory_agg = (
        inventory.where(F.col("adjustment_type").contains("Seasonally adjusted"))
                 .withColumn("month_date", F.col("observation_date").cast("date"))
                 .groupBy("month_date")
                 .agg(
                     F.avg("inventory_sales_ratio").alias("inventory_ratio"),
                     F.first("health_status").alias("inventory_health"),
                     F.first("recommendation").alias("inventory_recommendation"),
                 )
    )

    imports = dlt.read("gold_statcan_imports_monthly")
    imports_agg = (
        imports.withColumn("month_date", F.col("observation_date").cast("date"))
               .select(
                   "month_date",
                   F.col("total_import_value_cad").alias("ca_elec_imports"),
                   F.col("napcs_categories_count"),
               )
    )

    # ---- 4. WITS tariffs (annual -> monthly via forward fill) ----
    tariff = dlt.read("gold_wits_tariff_electronics")
    tariff_monthly = (
        tariff.withColumn("month_date", F.make_date(F.col("year"), F.lit(1), F.lit(1)))
              .select(
                  "month_date",
                  F.col("tariff_rate_pct").alias("elec_tariff_rate"),
                  F.col("yoy_pct_change").alias("tariff_yoy_pct"),
                  F.col("rolling_3yr_avg").alias("tariff_rolling_3yr"),
              )
    )

    # ---- 5. Join all sources on month_date ----
    joined = (
        fred_pivot
        .join(trends_monthly, on="month_date", how="left")
        .join(retail_agg, on="month_date", how="left")
        .join(inventory_agg, on="month_date", how="left")
        .join(imports_agg, on="month_date", how="left")
        .join(tariff_monthly, on="month_date", how="left")
    )

    # ---- 6. Forward-fill sparse columns (tariffs, monthly macro) ----
    ff_window = Window.orderBy("month_date").rowsBetween(Window.unboundedPreceding, 0)

    sparse_cols = [
        "elec_tariff_rate", "tariff_yoy_pct", "tariff_rolling_3yr",
        "freight_cass", "ppi_electronics", "retail_elec_sales",
        "consumer_sentiment", "housing_starts", "real_disposable_income",
        "ca_retail_elec_sales", "inventory_ratio", "ca_elec_imports",
    ]

    for c in sparse_cols:
        joined = joined.withColumn(
            c, F.last(F.col(c), ignorenulls=True).over(ff_window)
        )

    # ---- 7. Derived interaction features ----
    joined = (
        joined
        .withColumn(
            "import_to_sales_ratio",
            F.when(F.col("ca_retail_elec_sales").isNotNull() & (F.col("ca_retail_elec_sales") != 0),
                   F.col("ca_elec_imports") / F.col("ca_retail_elec_sales"))
        )
        .withColumn(
            "tariff_sentiment_interaction",
            F.col("elec_tariff_rate") * F.col("consumer_sentiment")
        )
        .withColumn(
            "demand_supply_gap",
            F.col("demand_trend_score") - (F.col("inventory_ratio") * 50)
        )
    )

    # ---- 8. Lag features for time-series ML ----
    w = Window.orderBy("month_date")

    lag_configs = {
        "ca_retail_elec_sales": [1, 3, 6, 12],
        "demand_trend_score": [1, 3],
        "inventory_ratio": [1, 3],
        "ca_elec_imports": [1, 3],
        "oil_wti": [1, 3],
        "fx_cad_usd": [1, 3],
    }

    for col_name, lags in lag_configs.items():
        for lag_n in lags:
            joined = joined.withColumn(
                f"{col_name}_lag{lag_n}m",
                F.lag(col_name, lag_n).over(w)
            )

    return joined.withColumn("ingest_ts", F.current_timestamp())